# DiffPIR — Deblur & Denoise
Metodo generativo basato su diffusion models per il restauro di immagini degradate.

**Setup richiesto:**
```bash
git clone https://github.com/yuanzhi-zhu/DiffPIR.git external/diffpir
mkdir -p src/methods/diffpir/weights
curl -L "https://openaipublic.blob.core.windows.net/diffusion/jul-2021/256x256_diffusion_uncond.pt" -o src/methods/diffpir/weights/256x256_diffusion_uncond.pt
```

In [1]:
import sys
sys.path.insert(0, '..')
import torch
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from tqdm import tqdm
import pandas as pd

from src.data.dataset import load_config, LBCDataset
from src.degradation.degradation import degrade
from src.methods.diffpir.diffpir import run_diffpir
from src.eval.metrics import evaluate

config = load_config()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

Device: cpu


## Configurazione

In [2]:
noise_levels = config['degradation']['noise_levels']
num_steps = config['diffpir']['num_steps']
kernel_size = config['degradation']['kernel_size']
blur_sigma = config['degradation']['blur_sigma']

print(f'Noise levels: {noise_levels}')
print(f'Num steps: {num_steps}')
print(f'Kernel: size={kernel_size}, sigma={blur_sigma}')

Noise levels: [0.005, 0.01, 0.05, 0.1]
Num steps: 20
Kernel: size=9, sigma=2


## Caricamento dataset di test

In [3]:
test_dataset = LBCDataset('data/splits/test.txt', config['dataset']['image_size'])
print(f'Immagini di test: {len(test_dataset)}')

Immagini di test: 145


## Demo su singole immagini
Esecuzione DiffPIR su alcune immagini di esempio per verifica visiva.

In [ ]:
n_examples = 3
noise_level = 0.05

fig, axes = plt.subplots(n_examples, 3, figsize=(12, 4 * n_examples))

for i in range(n_examples):
    gt = test_dataset[i]
    degraded = degrade(gt, kernel_size=kernel_size, sigma=blur_sigma, noise_level=noise_level)
    
    print(f'Processando immagine {i+1}/{n_examples}...')
    restored = run_diffpir(
        degraded.to(device),
        num_steps=num_steps,
        noise_level=noise_level,
        kernel_size=kernel_size,
        blur_sigma=blur_sigma,
    )
    
    def to_img(t):
        return (t * 0.5 + 0.5).clamp(0, 1).permute(1, 2, 0).numpy()
    
    axes[i, 0].imshow(to_img(gt))
    axes[i, 0].set_title('Ground Truth')
    axes[i, 0].axis('off')
    
    axes[i, 1].imshow(to_img(degraded))
    axes[i, 1].set_title(f'Degraded (noise={noise_level})')
    axes[i, 1].axis('off')
    
    axes[i, 2].imshow(to_img(restored))
    axes[i, 2].set_title('DiffPIR Restored')
    axes[i, 2].axis('off')

plt.tight_layout()
plt.show()

Processando immagine 1/3...
Processando immagine 2/3...
Processando immagine 3/3...


## Valutazione quantitativa
Calcolo PSNR e SSIM su un sottoinsieme del test set per tutti i livelli di rumore.

In [ ]:
n_eval = 10
results = []

for noise_level in noise_levels:
    psnr_list, ssim_list, time_list = [], [], []
    
    for i in tqdm(range(n_eval), desc=f'noise={noise_level}'):
        gt = test_dataset[i]
        degraded = degrade(gt, kernel_size=kernel_size, sigma=blur_sigma, noise_level=noise_level)
        
        restored, inference_time = run_diffpir(
            degraded.to(device),
            num_steps=num_steps,
            noise_level=noise_level,
            kernel_size=kernel_size,
            blur_sigma=blur_sigma,
            return_timing=True,
        )
        
        m = evaluate(restored, gt)
        psnr_list.append(m['psnr'])
        ssim_list.append(m['ssim'])
        time_list.append(inference_time)
    
    avg_psnr = np.mean(psnr_list)
    avg_ssim = np.mean(ssim_list)
    avg_time = np.mean(time_list)
    results.append({
        'method': 'diffpir',
        'noise_level': noise_level,
        'psnr': avg_psnr,
        'ssim': avg_ssim,
        'avg_inference_time': avg_time
    })
    print(f'[DiffPIR] noise={noise_level} | PSNR={avg_psnr:.2f} | SSIM={avg_ssim:.4f} | Time={avg_time:.1f}s')

df_results = pd.DataFrame(results)
df_results

## Salvataggio risultati

In [ ]:
results_dir = Path(config['eval']['results_dir'])
results_dir.mkdir(parents=True, exist_ok=True)

df_results.to_csv(results_dir / 'diffpir_results.csv', index=False)
print(f'Risultati salvati in {results_dir / "diffpir_results.csv"}')

## Confronto con altri metodi
Caricamento risultati TV e UNet per confronto finale.

In [ ]:
results_dir = Path(config['eval']['results_dir'])

df_tv = pd.read_csv(results_dir / 'tv_results.csv') if (results_dir / 'tv_results.csv').exists() else None
df_unet = pd.read_csv(results_dir / 'unet_results.csv') if (results_dir / 'unet_results.csv').exists() else None
df_diffpir = pd.read_csv(results_dir / 'diffpir_results.csv')

dfs = [df for df in [df_tv, df_unet, df_diffpir] if df is not None]
if dfs:
    df_all = pd.concat(dfs, ignore_index=True)
    print(df_all.pivot(index='noise_level', columns='method', values=['psnr', 'ssim']))
else:
    print('Esegui prima TV e UNet per il confronto completo.')